In [1]:
import numpy as np
import pandas as pd
from astroquery.gaia import Gaia
import warnings
warnings.filterwarnings('ignore')

lcgen = pd.read_csv('cf_data/dr3_xmatch_from_lcgen.csv', dtype={'GaiaDR3_ID': 'Int64'})
print(lcgen.shape)
print(lcgen['GaiaDR3_ID'].duplicated().sum())
print(lcgen['GaiaDR3_ID'].isna().sum())
print(lcgen.head())

The archive is unstable and may perform below expectations. Please avoid launching intense Python query showers. Please contact the Gaia helpdesk in case of questions (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk). Workaround solutions for the issues following the recent infrastructure upgrade: https://www.cosmos.esa.int/web/gaia/news#WorkaroundArchive
(8176, 11)
358
41
                object source_survey_name     source_survey_id        ref  \
0   109628013733250304              gaia2   109628013733250304  Kiman2021   
1   128095411034209536              gaia2   128095411034209536  Kiman2021   
2  1321738561431758592              gaia2  1321738561431758592  Kiman2021   
3   132363027978672000              gaia2   132363027978672000  Kiman2021   
4   132592688469662208              gaia2   132592688469662208  Kiman2021   

   Prot  e_Prot  ageMyr  e_ageMyr       TIC_ID           GaiaDR2_ID  \
0   NaN     NaN    24.0       2.0  371422484.0   109628013733250304   
1   NaN     NaN  

In [2]:
from astroquery.simbad import Simbad

simbad = Simbad()
simbad.add_votable_fields('ids')

missing = lcgen[lcgen['GaiaDR3_ID'].isna() & lcgen['GaiaDR2_ID'].isna()].copy()

recovered = []
for _, row in missing.iterrows():
    try:
        result = simbad.query_object(row['object'])
        if result is None:
            continue
        ids = result['ids'][0]
        for id_str in ids.split('|'):
            if 'Gaia DR3' in id_str:
                dr3_id = int(id_str.replace('Gaia DR3', '').strip())
                recovered.append({'object': row['object'], 'GaiaDR3_ID': dr3_id})
                break
    except:
        continue

recovered_df = pd.DataFrame(recovered)
print(f'Recovered {len(recovered_df)} DR3 IDs from SIMBAD')
print(recovered_df)

Recovered 24 DR3 IDs from SIMBAD
                     object           GaiaDR3_ID
0   2MASS J06170531+8353354  1149897160435827200
1   2MASS J07473239+4808438   933708229245319552
2   2MASS J07473462+4807300   933708164821511424
3   2MASS J15421300+6537051  1641502179449156096
4   2MASS J15483685-5045256  5982345155045648640
5   2MASS J15483762-5045143  5982345155045187584
6   2MASS J21005492-4131438  6773423224070637952
7   2MASS J21010380-4114331  6773453838597541760
8   2MASS J23095781+5506472  1996725077535282944
9             TIC 129117325  5905821894507108864
10            TIC 327150358  5855593454594292736
11            TIC 333670784  6126469654585981952
12            TIC 334124479  6125358151409872256
13            TIC 357019659  5199981334076455680
14            TIC 382288432  6054488059982875392
15            TIC 390294856  5241435980496897280
16            TIC 429235355  6126689694347231360
17            TIC 448852739  5846624596475789056
18            TIC 458181603  5970999

In [3]:
recovered_df = recovered_df.set_index('object')
lcgen.loc[lcgen['object'].isin(recovered_df.index), 'GaiaDR3_ID'] = \
    lcgen.loc[lcgen['object'].isin(recovered_df.index), 'object'].map(recovered_df['GaiaDR3_ID'])

print(f'Missing DR3 IDs remaining: {lcgen["GaiaDR3_ID"].isna().sum()}')

Missing DR3 IDs remaining: 17


In [4]:
lcgen_valid = lcgen.dropna(subset=['GaiaDR3_ID']).copy()
ids = lcgen_valid['GaiaDR3_ID'].drop_duplicates().tolist()
print(f'Unique IDs to query: {len(ids)}')

Unique IDs to query: 7832


In [5]:
from tqdm.notebook import tqdm

batch_size = 1000
all_results = []

batches = [ids[i:i+batch_size] for i in range(0, len(ids), batch_size)]

for batch in tqdm(batches, desc='Querying Gaia'):
    ids_str = ', '.join(str(i) for i in batch)
    query = f"""
    SELECT source_id, bp_rp, ebpminrp_gspphot, ruwe, parallax
    FROM gaiadr3.gaia_source
    WHERE source_id IN ({ids_str})
    """
    try:
        result = Gaia.launch_job(query).get_results().to_pandas()
        all_results.append(result)
    except:
        print(f'Batch failed, skipping')

gaia_results = pd.concat(all_results).reset_index(drop=True)
gaia_results['source_id'] = gaia_results['source_id'].astype('Int64')
print(f'Total Gaia results: {len(gaia_results)} rows')

Querying Gaia:   0%|          | 0/8 [00:00<?, ?it/s]

Total Gaia results: 7832 rows


In [7]:
lcgen = lcgen_valid.merge(
    gaia_results.rename(columns={'source_id': 'GaiaDR3_ID'}),
    on='GaiaDR3_ID',
    how='left'
)

# compute bprp0
lcgen['bprp0'] = lcgen['bp_rp'] - lcgen['ebpminrp_gspphot']

print(f'Merged: {len(lcgen)} rows')
print(f'Missing bprp0: {lcgen["bprp0"].isna().sum()}')

lcgen.to_csv('cf_data/dr3_xmatch_from_lcgen.csv', index=False)
print('Saved!')

Merged: 8159 rows
Missing bprp0: 1881
Saved!


In [3]:
import pandas as pd 
lcgen = pd.read_csv('cf_data/dr3_xmatch_from_lcgen.csv', dtype={'GaiaDR3_ID': 'Int64'})
dups = lcgen[lcgen['GaiaDR3_ID'].duplicated(keep=False)]
print(dups[['GaiaDR3_ID', 'ref', 'ageMyr', 'Prot']].sort_values('GaiaDR3_ID').head(20))

             GaiaDR3_ID            ref  ageMyr   Prot
569   47917816253918720      Kiman2021   750.0    NaN
1289  47917816253918720           LWRD   432.0  10.95
2052  53335045618385536  Feinstein2024   127.4    NaN
632   53335045618385536      Kiman2021   112.0    NaN
7233  63485427727244544  Feinstein2024   127.4    NaN
856   63485427727244544      Kiman2021   112.0    NaN
870   64006905476619776      Kiman2021   112.0    NaN
6305  64006905476619776  Feinstein2024   127.4    NaN
6413  64050645425295872  Feinstein2024   127.4    NaN
872   64050645425295872      Kiman2021   112.0    NaN
875   64094037476647040      Kiman2021   112.0    NaN
6417  64094037476647040  Feinstein2024   127.4    NaN
1468  64438841747360768  Feinstein2024   127.4    NaN
878   64438841747360768      Kiman2021   112.0    NaN
879   64445988572906752      Kiman2021   112.0    NaN
1410  64445988572906752  Feinstein2024   127.4    NaN
1361  64597927335800064  Feinstein2024   127.4    NaN
883   64597927335800064     

In [4]:
missing_lcgen = lcgen[lcgen['GaiaDR3_ID'].isna()]
print(missing_lcgen[['object', 'ref']].to_string())

Empty DataFrame
Columns: [object, ref]
Index: []


In [5]:
from astroquery.simbad import Simbad
from astroquery.gaia import Gaia

simbad = Simbad()
simbad.add_votable_fields('ids')

unrecovered_objects = [
    '2MASS J16243140+4549570', 'TIC 121102484', 'TIC 121672600',
    'TIC 146559012', 'TIC 159153860', 'TIC 171611328', 'TIC 241398115',
    'TIC 312093689', 'TIC 329162201', 'TIC 405567821', 'TIC 440497672',
    'TIC 72178247', 'TIC 72858760'
]

recovered2 = []
for obj in unrecovered_objects:
    try:
        result = simbad.query_object(obj)
        if result is None:
            print(f'Not found: {obj}')
            continue
        ids = result['ids'][0]
        for id_str in ids.split('|'):
            if 'Gaia DR3' in id_str:
                dr3_id = int(id_str.replace('Gaia DR3', '').strip())
                recovered2.append({'object': obj, 'GaiaDR3_ID': dr3_id})
                print(f'Recovered: {obj} -> {dr3_id}')
                break
        else:
            print(f'No DR3 ID found: {obj}')
    except Exception as e:
        print(f'Error for {obj}: {e}')

print(f'\nTotal newly recovered: {len(recovered2)}')

The archive is unstable and may perform below expectations. Please avoid launching intense Python query showers. Please contact the Gaia helpdesk in case of questions (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk). Workaround solutions for the issues following the recent infrastructure upgrade: https://www.cosmos.esa.int/web/gaia/news#WorkaroundArchive
No DR3 ID found: 2MASS J16243140+4549570
No DR3 ID found: TIC 121102484
No DR3 ID found: TIC 121672600
No DR3 ID found: TIC 146559012
No DR3 ID found: TIC 159153860
No DR3 ID found: TIC 171611328
No DR3 ID found: TIC 241398115


Error for TIC 312093689: index 0 is out of bounds for axis 0 with size 0
No DR3 ID found: TIC 329162201
No DR3 ID found: TIC 405567821
No DR3 ID found: TIC 440497672
Error for TIC 72178247: index 0 is out of bounds for axis 0 with size 0
No DR3 ID found: TIC 72858760

Total newly recovered: 0


In [6]:
print(lcgen[lcgen['bprp0'].isna()][['bp_rp', 'ebpminrp_gspphot']].isna().sum())

bp_rp                  7
ebpminrp_gspphot    1881
dtype: int64
